# Qwen3.5-9B From Scratch

## Architecture

                                  [Inputs]
                                      │
                                      ▼
                             ┌──────────────────┐
                             │  Embedding Layer │
                             └──────────────────┘
                                      │
                                      ▼
        ┌─────────────────────────────────────────────────────────────┐
        │ 8× Repeating Macro-Blocks (32 Layers Total)                 │
        │                                                             │
        │  ┌───────────────────────────────────────────────────────┐  │
        │  │ 3× Linear Attention Layers                            │  │
        │  │    [Gated DeltaNet Block] ──► [SiLU-Gated FFN Block]  │  │
        │  └───────────────────────────────────────────────────────┘  │
        │                             │                               │
        │                             ▼                               │
        │  ┌───────────────────────────────────────────────────────┐  │
        │  │ 1× Full Attention Layer                               │  │
        │  │    [Gated Attention (GQA)] ──► [SiLU-Gated FFN Block] │  │
        │  └───────────────────────────────────────────────────────┘  │
        └─────────────────────────────────────────────────────────────┘
                                      │
                                      ▼
                             ┌──────────────────┐
                             │     RMSNorm      │
                             └──────────────────┘
                                      │
                                      ▼
                            [Next tokens / Outputs]

## 1. RMSNorm


In [1]:
from torch import nn
import torch
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Qwen3.5 uses (1 + weight) scaling with zero init
        self.weight = nn.Parameter(torch.zeros(emb_dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

    def forward(self, x):
        x_norm = self._norm(x.float())
        x_norm = x_norm * (1.0 + self.weight.float())
        return x_norm.to(dtype=x.dtype)

class RMSNormGated(nn.Module):
    """
    Element-wise SiLU gated RMSNorm.
    Qwen3_5RMSNormGated from the HF implementation.
    """
    def __init__(self, n_embed, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embed))
        self.variance_epsilon = eps

    def forward(self, x, gate):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        variance = x.pow(2).mean(-1, keepdim=True)
        # Norm before gate
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        x = self.weight * x.to(input_dtype)
        return (x * F.silu(gate.to(torch.float32))).to(input_dtype)

## 2. Rotary Positional Embedding (RoPE)

In [2]:
def compute_rope_params(
    head_dim,
    theta_base=10_000,
    context_length=4096,
    partial_rotary_factor=1.0,
    dtype=torch.float32,
):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    rotary_dim = int(head_dim * partial_rotary_factor)
    rotary_dim = max(2, rotary_dim - (rotary_dim % 2))

    inv_freq = 1.0 / (
        theta_base ** (
            torch.arange(0, rotary_dim, 2, dtype=dtype)[: (rotary_dim // 2)].float() / rotary_dim
        )
    )

    positions = torch.arange(context_length, dtype=dtype)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    angles = torch.cat([angles, angles], dim=1)

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    _, _, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    rot_dim = cos.shape[-1]
    if rot_dim > head_dim:
        raise ValueError(f"RoPE dim {rot_dim} cannot exceed head_dim {head_dim}.")

    x_rot = x[..., :rot_dim]
    x_pass = x[..., rot_dim:]

    x1 = x_rot[..., : rot_dim // 2]
    x2 = x_rot[..., rot_dim // 2 :]

    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x_rot * cos) + (rotated * sin)

    x_out = torch.cat([x_rotated, x_pass], dim=-1)
    return x_out.to(dtype=x.dtype)

## 3. Full Attention (Grouped Query Attention)

In [3]:
from einops import rearrange
from torch import einsum

class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention with partial RoPE and optional sigmoid output gate.
    Used as the 'full_attention' layer in Qwen3.5 (every 4th layer).
    Qwen3.5-9B: n_heads=16, num_kv_groups=4, head_dim=256.
    """

    def __init__(self, idim, n_heads, num_kv_groups, head_dim, dtype):
        super().__init__()
        self.idim = idim
        self.n_heads = n_heads
        self.num_kv_groups = num_kv_groups
        self.head_dim = head_dim
        self.group_size = n_heads // num_kv_groups
        self.odim = n_heads * head_dim
        self.scale = head_dim ** -0.5

        # Q and output gate are tied into a single 2× projection (HF convention).
        self.W_query = nn.Linear(idim, self.odim * 2, dtype=dtype, bias=False)
        self.k_proj = nn.Linear(idim, head_dim * num_kv_groups, dtype=dtype, bias=False)
        self.v_proj = nn.Linear(idim, head_dim * num_kv_groups, dtype=dtype, bias=False)
        self.o_proj = nn.Linear(self.odim, idim, dtype=dtype, bias=False)

        self.q_norm = RMSNorm(head_dim, eps=1e-6)
        self.k_norm = RMSNorm(head_dim, eps=1e-6)

    def forward(self, x, cos, sin, mask=None):
        b, L, _ = x.shape

        # Combined Q + gate projection (2× linear), then split
        q_raw = self.W_query(x)                              # (B, L, odim * 2)
        q_raw = q_raw.view(b, L, self.n_heads, self.head_dim * 2)
        q, gate = torch.chunk(q_raw, 2, dim=-1)              # each (B, L, H, head_dim)
        gate = gate.reshape(b, L, self.odim)                 # (B, L, odim)
        q = q.transpose(1, 2)                                # (B, H, L, head_dim)

        k = self.k_proj(x)   # (B, L, head_dim * num_kv_groups)
        v = self.v_proj(x)   # (B, L, head_dim * num_kv_groups)
        k = rearrange(k, 'b l (g d) -> b g l d', g=self.num_kv_groups)
        v = rearrange(v, 'b l (g d) -> b g l d', g=self.num_kv_groups)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        dots = dots.masked_fill(mask, -torch.inf)
        attn = dots.softmax(dim=-1)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h l d -> b l (h d)')

        # Qwen3.5 full-attention uses a gated Q projection
        out = out * torch.sigmoid(gate)
        return self.o_proj(out)

## 4. Linear Attention (Gated DeltaNet Attention)

In [4]:
def l2norm(x, dim=-1, eps=1e-6):
    """Unit L2 normalisation without a learnable scale (matches HF convention)."""
    return x * torch.rsqrt((x * x).sum(dim=dim, keepdim=True) + eps)

def torch_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=64,
):
    initial_dtype = query.dtype
    query = l2norm(query, dim=-1, eps=1e-6)
    key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    # reshape to chunks
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    # chunk decay
    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    for i in range(1, chunk_size):
        row = attn[..., i, :i].clone()
        sub = attn[..., :i, :i].clone()
        attn[..., i, :i] = row + (row.unsqueeze(-1) * sub).sum(-2)
    attn = attn + torch.eye(chunk_size, dtype=attn.dtype, device=attn.device)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim).to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    # for each chunk
    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = (q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]).masked_fill_(mask, 0)
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out

class GatedDeltaNetAttention(nn.Module):
    """
    Gated DeltaNet linear attention, tailored from HF Qwen3_5GatedDeltaNet.
    Qwen3.5-9B config: num_key_heads=16, num_value_heads=32, key_head_dim=128, value_head_dim=128, conv_kernel_dim=4.
    """
    def __init__(self, idim, num_key_heads, num_value_heads, key_head_dim, value_head_dim, conv_kernel_dim, dtype):
        super().__init__()
        self.num_k_heads = num_key_heads
        self.num_v_heads = num_value_heads
        self.head_k_dim  = key_head_dim
        self.head_v_dim  = value_head_dim
        self.key_dim     = key_head_dim * num_key_heads
        self.value_dim   = value_head_dim * num_value_heads

        self.in_proj_qkv = nn.Linear(idim, self.key_dim * 2 + self.value_dim, dtype=dtype, bias=False)
        self.in_proj_z   = nn.Linear(idim, self.value_dim, dtype=dtype, bias=False)
        self.in_proj_b   = nn.Linear(idim, num_value_heads, dtype=dtype, bias=False)
        self.in_proj_a   = nn.Linear(idim, num_value_heads, dtype=dtype, bias=False)

        # QKV
        conv_dim = self.key_dim * 2 + self.value_dim
        self.conv1d = nn.Conv1d(conv_dim, conv_dim, kernel_size=conv_kernel_dim,
                                padding=conv_kernel_dim - 1, groups=conv_dim,
                                dtype=dtype, bias=False)

        self.dt_bias = nn.Parameter(torch.ones(num_value_heads, dtype=dtype))

        A = torch.empty(num_value_heads).uniform_(0, 16)
        self.A_log = nn.Parameter(torch.log(A))

        # RMSNorm gated by silu(z), normalises per value-head dimension
        self.norm = RMSNormGated(value_head_dim, eps=1e-6)

        self.out_proj = nn.Linear(self.value_dim, idim, dtype=dtype, bias=False)

    def forward(self, hidden_states):
        # Set up dimensions for reshapes later
        batch_size, seq_len, _ = hidden_states.shape

        mixed_qkv = self.in_proj_qkv(hidden_states)
        mixed_qkv = mixed_qkv.transpose(1, 2)

        z = self.in_proj_z(hidden_states)
        z = z.reshape(batch_size, seq_len, -1, self.head_v_dim)

        b = self.in_proj_b(hidden_states)
        a = self.in_proj_a(hidden_states)

        mixed_qkv = F.silu(self.conv1d(mixed_qkv)[:, :, :seq_len])

        mixed_qkv = mixed_qkv.transpose(1, 2)
        query, key, value = torch.split(
            mixed_qkv,
            [
                self.key_dim,
                self.key_dim,
                self.value_dim,
            ],
            dim=-1,
        )

        query = query.reshape(batch_size, seq_len, -1, self.head_k_dim)
        key = key.reshape(batch_size, seq_len, -1, self.head_k_dim)
        value = value.reshape(batch_size, seq_len, -1, self.head_v_dim)

        beta = b.sigmoid()
        # If the model is loaded in fp16, without the .float() here, A might be -inf
        g = -self.A_log.float().exp() * F.softplus(a.float() + self.dt_bias)
        if self.num_v_heads // self.num_k_heads > 1:
            query = query.repeat_interleave(self.num_v_heads // self.num_k_heads, dim=2)
            key = key.repeat_interleave(self.num_v_heads // self.num_k_heads, dim=2)

        core_attn_out = torch_chunk_gated_delta_rule(
            query,
            key,
            value,
            g=g,
            beta=beta,
        )

        # reshape input data into 2D tensor
        core_attn_out = core_attn_out.reshape(-1, self.head_v_dim)
        z = z.reshape(-1, self.head_v_dim)
        core_attn_out = self.norm(core_attn_out, z)
        core_attn_out = core_attn_out.reshape(batch_size, seq_len, -1)

        output = self.out_proj(core_attn_out)
        return output

## 5. Qwen3.5 Building Blocks


In [5]:
class GatedFeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim, dtype):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim, hidden_dim, dtype=dtype, bias=False)
        self.fc2 = nn.Linear(emb_dim, hidden_dim, dtype=dtype, bias=False)
        self.fc3 = nn.Linear(hidden_dim, emb_dim, dtype=dtype, bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = F.silu(x_fc1) * x_fc2
        return self.fc3(x)

class GatedAttentionBlock(nn.Module):
    """
    Full (quadratic-softmax) GQA block with sigmoid output gate.
    Instantiated every 4th layer (full_attention_interval=4).
    Qwen3.5-9B: n_heads=16, num_kv_groups=4, head_dim=256.
    """
    def __init__(self, dim, n_heads, num_kv_groups, head_dim, mlp_dim, dtype):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = GroupedQueryAttention(dim, n_heads=n_heads, num_kv_groups=num_kv_groups,
                                           head_dim=head_dim, dtype=dtype)
        self.norm2 = RMSNorm(dim)
        self.ff    = GatedFeedForward(dim, mlp_dim, dtype)

    def forward(self, x, cos, sin, mask=None):
        x = self.attn(self.norm1(x), cos, sin, mask) + x
        x = self.ff(self.norm2(x)) + x
        return x

class GatedDeltaNetBlock(nn.Module):
    """
    Linear-attention DeltaNet block with SiLU output gate.
    Occupies the 3 layers preceding each GatedAttentionBlock.
    Qwen3.5-9B: num_key_heads=16, num_value_heads=32,
                key_head_dim=128, value_head_dim=128, conv_kernel_dim=4.
    """
    def __init__(self, dim, num_key_heads, num_value_heads, key_head_dim, value_head_dim,
                 conv_kernel_dim, mlp_dim, dtype):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = GatedDeltaNetAttention(dim,
                                            num_key_heads=num_key_heads,
                                            num_value_heads=num_value_heads,
                                            key_head_dim=key_head_dim,
                                            value_head_dim=value_head_dim,
                                            conv_kernel_dim=conv_kernel_dim,
                                            dtype=dtype)
        self.norm2 = RMSNorm(dim)
        self.ff    = GatedFeedForward(dim, mlp_dim, dtype)

    def forward(self, x, cos=None, sin=None, mask=None):
        # DeltaNet does not use positional encoding; cos/sin kept for a
        # uniform block interface so the model loop can call all blocks alike.
        x = self.attn(self.norm1(x)) + x
        x = self.ff(self.norm2(x)) + x
        return x

## 6. Qwen3.5 Model

In [6]:
class Qwen3_5Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        layer_types = cfg.get("layer_types", ["full_attention"] * cfg["n_layers"])
        if len(layer_types) != cfg["n_layers"]:
            raise ValueError("len(layer_types) must equal n_layers")

        self.trf_blocks = nn.ModuleList()
        for ltype in layer_types:
            if ltype == 'full_attention':
                self.trf_blocks.append(
                    GatedAttentionBlock(cfg["emb_dim"], cfg["n_heads"], cfg["n_kv_groups"], cfg["head_dim"],
                                        cfg["hidden_dim"], cfg["dtype"]))
            else:  # 'linear_attention'
                self.trf_blocks.append(
                    GatedDeltaNetBlock(cfg["emb_dim"],
                                       cfg["linear_num_key_heads"], cfg["linear_num_value_heads"],
                                       cfg["linear_key_head_dim"], cfg["linear_value_head_dim"],
                                       cfg["linear_conv_kernel_dim"],
                                       cfg["hidden_dim"], cfg["dtype"]))

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        head_dim = cfg["emb_dim"] // cfg["n_heads"] if cfg["head_dim"] is None else cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            partial_rotary_factor=cfg.get("partial_rotary_factor", 1.0),
            dtype=torch.float32,
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.dtype = cfg["dtype"]

    def forward(self, in_idx):
        x = self.tok_emb(in_idx)

        num_tokens = x.shape[1]
        mask = torch.triu(
            torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool),
            diagonal=1,
        )

        for block in self.trf_blocks:
            x = block(x, self.cos, self.sin, mask)

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.dtype))
        return logits

In [7]:
# Qwen3.5-9B text configuration
QWEN3_5_CONFIG = {
    # General
    "vocab_size": 248_320,
    "context_length": 1_010_000,
    "emb_dim": 4_096,
    "rms_norm_eps": 1e-6,
    # Full attention (GroupedQueryAttention)
    "n_heads": 16,
    "n_kv_groups": 4,
    "head_dim": 256,
    "rope_base": 10_000_000.0,
    "partial_rotary_factor": 0.25,
    # Linear attention (GatedDeltaNetAttention)
    "linear_num_key_heads": 16,
    "linear_num_value_heads": 32,
    "linear_key_head_dim": 128,
    "linear_value_head_dim": 128,
    "linear_conv_kernel_dim": 4,
    # Common
    "n_layers": 32,
    "hidden_dim": 12_288,
    "layer_types": [
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
    ],
    "dtype": torch.bfloat16,
}

In [8]:
model = Qwen3_5Model(QWEN3_5_CONFIG)
print("\nModel : \n", model)


Model : 
 Qwen3_5Model(
  (tok_emb): Embedding(248320, 4096)
  (trf_blocks): ModuleList(
    (0-2): 3 x GatedDeltaNetBlock(
      (norm1): RMSNorm()
      (attn): GatedDeltaNetAttention(
        (in_proj_qkv): Linear(in_features=4096, out_features=8192, bias=False)
        (in_proj_z): Linear(in_features=4096, out_features=4096, bias=False)
        (in_proj_b): Linear(in_features=4096, out_features=32, bias=False)
        (in_proj_a): Linear(in_features=4096, out_features=32, bias=False)
        (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
        (norm): RMSNormGated()
        (out_proj): Linear(in_features=4096, out_features=4096, bias=False)
      )
      (norm2): RMSNorm()
      (ff): GatedFeedForward(
        (fc1): Linear(in_features=4096, out_features=12288, bias=False)
        (fc2): Linear(in_features=4096, out_features=12288, bias=False)
        (fc3): Linear(in_features=12288, out_features=4096, bias=False)
      )
    )

In [9]:
# A quick check that the forward pass works before continuing:
test = model(torch.tensor([1, 2, 3]).unsqueeze(0))
print("Test model output shape : ", test.shape)

Test model output shape :  torch.Size([1, 3, 248320])


In [10]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

Total number of parameters: 8,953,803,264

Total number of unique parameters: 7,936,684,544


In [11]:
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

float32 (PyTorch default): 67.19 GB
bfloat16: 33.60 GB


In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

## 7. Load pretrained weights

In [13]:
def load_weights_into_qwen3_5(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    if "model.embed_tokens.weight" in params:
        model_prefix = "model"
    elif "model.language_model.embed_tokens.weight" in params:
        model_prefix = "model.language_model"
    else:
        raise KeyError("Could not find embed token weights in checkpoint.")

    def pkey(suffix):
        return f"{model_prefix}.{suffix}"

    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params[pkey("embed_tokens.weight")],
        pkey("embed_tokens.weight"),
    )

    n_layers = param_config["n_layers"]
    layer_types = param_config.get("layer_types", ["full_attention"] * n_layers)

    for l in range(n_layers):
        block = model.trf_blocks[l]
        layer_type = layer_types[l]

        if layer_type == "full_attention":
            att = block.attn
            att.W_query.weight = assign(
                att.W_query.weight,
                params[pkey(f"layers.{l}.self_attn.q_proj.weight")],
                pkey(f"layers.{l}.self_attn.q_proj.weight"),
            )
            att.k_proj.weight = assign(
                att.k_proj.weight,
                params[pkey(f"layers.{l}.self_attn.k_proj.weight")],
                pkey(f"layers.{l}.self_attn.k_proj.weight"),
            )
            att.v_proj.weight = assign(
                att.v_proj.weight,
                params[pkey(f"layers.{l}.self_attn.v_proj.weight")],
                pkey(f"layers.{l}.self_attn.v_proj.weight"),
            )
            att.o_proj.weight = assign(
                att.o_proj.weight,
                params[pkey(f"layers.{l}.self_attn.o_proj.weight")],
                pkey(f"layers.{l}.self_attn.o_proj.weight"),
            )
            if hasattr(att, "q_norm") and att.q_norm is not None:
                att.q_norm.weight = assign(
                    att.q_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.q_norm.weight")],
                    pkey(f"layers.{l}.self_attn.q_norm.weight"),
                )
            if hasattr(att, "k_norm") and att.k_norm is not None:
                att.k_norm.weight = assign(
                    att.k_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.k_norm.weight")],
                    pkey(f"layers.{l}.self_attn.k_norm.weight"),
                )

        elif layer_type == "linear_attention":
            lat = block.attn
            lat.dt_bias = assign(
                lat.dt_bias,
                params[pkey(f"layers.{l}.linear_attn.dt_bias")],
                pkey(f"layers.{l}.linear_attn.dt_bias"),
            )
            lat.A_log = assign(
                lat.A_log,
                params[pkey(f"layers.{l}.linear_attn.A_log")],
                pkey(f"layers.{l}.linear_attn.A_log"),
            )
            lat.conv1d.weight = assign(
                lat.conv1d.weight,
                params[pkey(f"layers.{l}.linear_attn.conv1d.weight")],
                pkey(f"layers.{l}.linear_attn.conv1d.weight"),
            )
            lat.norm.weight = assign(
                lat.norm.weight,
                params[pkey(f"layers.{l}.linear_attn.norm.weight")],
                pkey(f"layers.{l}.linear_attn.norm.weight"),
            )
            lat.out_proj.weight = assign(
                lat.out_proj.weight,
                params[pkey(f"layers.{l}.linear_attn.out_proj.weight")],
                pkey(f"layers.{l}.linear_attn.out_proj.weight"),
            )
            lat.in_proj_qkv.weight = assign(
                lat.in_proj_qkv.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight"),
            )
            lat.in_proj_z.weight = assign(
                lat.in_proj_z.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_z.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_z.weight"),
            )
            lat.in_proj_b.weight = assign(
                lat.in_proj_b.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_b.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_b.weight"),
            )
            lat.in_proj_a.weight = assign(
                lat.in_proj_a.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_a.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_a.weight"),
            )

        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        block.norm1.weight = assign(
            block.norm1.weight,
            params[pkey(f"layers.{l}.input_layernorm.weight")],
            pkey(f"layers.{l}.input_layernorm.weight"),
        )

        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[pkey(f"layers.{l}.mlp.gate_proj.weight")],
            pkey(f"layers.{l}.mlp.gate_proj.weight"),
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[pkey(f"layers.{l}.mlp.up_proj.weight")],
            pkey(f"layers.{l}.mlp.up_proj.weight"),
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[pkey(f"layers.{l}.mlp.down_proj.weight")],
            pkey(f"layers.{l}.mlp.down_proj.weight"),
        )
        block.norm2.weight = assign(
            block.norm2.weight,
            params[pkey(f"layers.{l}.post_attention_layernorm.weight")],
            pkey(f"layers.{l}.post_attention_layernorm.weight"),
        )

    model.final_norm.weight = assign(
        model.final_norm.weight,
        params[pkey("norm.weight")],
        pkey("norm.weight"),
    )

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    elif pkey("lm_head.weight") in params:
        model.out_head.weight = assign(model.out_head.weight, params[pkey("lm_head.weight")], pkey("lm_head.weight"))
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

In [14]:
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

repo_id = "Qwen/Qwen3.5-9B"
local_dir = Path(repo_id).parts[-1]

repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

load_weights_into_qwen3_5(model, QWEN3_5_CONFIG, weights_dict)
model.to(device)
del weights_dict

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

## 8. Load tokenizer

In [15]:
import re
from pathlib import Path
from tokenizers import Tokenizer

class Qwen3_5Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(
        self,
        tokenizer_file_path="tokenizer.json",
        repo_id=None,
        apply_chat_template=True,
        add_generation_prompt=False,
        add_thinking=False,
    ):
        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        # Mirrors Qwen3.5 chat_template behavior:
        # add_generation_prompt + thinking => "<think>\n"
        # add_generation_prompt + no thinking => empty think scaffold
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant\n"
            if self.add_thinking:
                s += "<think>\n"
            else:
                s += "<think>\n\n</think>\n\n"
        return s

In [16]:
tokenizer_file_path = "Qwen3.5-9B/tokenizer.json"

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

tokenizer = Qwen3_5Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)

## 9. Generate text

In [17]:
def sample_next_token(logits, temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0, recent_tokens=None):
    """
    Apply temperature, top-k, top-p, and repetition penalty sampling.
    """

    # Temperature scaling
    if temperature != 1.0:
        logits = logits / temperature

    # Repetition penalty (penalize tokens that appeared recently)
    if repetition_penalty != 1.0 and recent_tokens is not None:
        for token in set(recent_tokens.tolist()):
            logits[..., token] /= repetition_penalty

    # Top-k filtering
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        values, indices = torch.topk(logits, top_k)
        mask = logits < values[..., -1, None]
        logits = logits.masked_fill(mask, float("-inf"))

    # Top-p (nucleus) filtering
    if top_p is not None:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(probs, dim=-1)

        # Mask tokens where cumulative prob > top_p
        sorted_mask = cumulative_probs > top_p
        # shift mask so at least 1 token is kept
        sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
        sorted_mask[..., 0] = False

        # apply mask
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        logits = logits.masked_fill(mask, float("-inf"))

    # Convert to probabilities
    probs = F.softmax(logits, dim=-1)

    # Sample
    next_token = torch.multinomial(probs, num_samples=1)
    return next_token

def generate_tokens(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=1.0,
    top_k=None,
    top_p=None,
    repetition_penalty=1.0,
    window_size=50,
):
    """
    Streaming text generation with advanced sampling.
    """
    model.eval()
    recent_tokens = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(token_ids)[:, -1, :]   # logits of last token
            next_token = sample_next_token(
                out,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                recent_tokens=token_ids[:, -window_size:].flatten()
            )

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            # Update sequence
            token_ids = torch.cat([token_ids, next_token], dim=1)
            recent_tokens.append(next_token.item())

In [18]:
import time
from datetime import timedelta

prompt = "What is world order and what is the trend?"
input_token_ids = tokenizer.encode(prompt)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)
prompt_text = tokenizer.decode(input_token_ids)
print(f"Prompt Text: {prompt_text}")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_tokens(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=8000,
    eos_token_id=tokenizer.eos_token_id,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.2,
    window_size=1024
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
# Convert and print as hour:minute:second
formatted_time = str(timedelta(seconds=elapsed))
print(f"\n\n\nTotal time: {formatted_time}")

tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"Generation speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")
else:
    print("No GPU available.")


Prompt Text: <|im_start|>user
What is world order and what is the trend?<|im_end|>
<|im_start|>assistant
<think>

Here's a thinking process that leads to the suggested answer:

1.  **Deconstruct the Request:**
    *   **Topic:** World Order (International System/Political Structure).
    *   **Questions:** What is it? What is the current trend/direction?
    *   **Goal:** Provide a comprehensive, balanced, and up-to-date overview of international relations theory regarding global stability/power dynamics.

2.  **Part 1: Defining "World Order"**
    *   *Concept:* It's not just one thing; it's a complex system. Needs definitions from multiple angles (political science, sociology, geopolitics).
    *   *Key Elements:* Hierarchy vs. Anarchy (Realism), Norms/Rules (Liberalism/IPE), Hegemon(s) (Who holds power?), Institutions (UN, WTO, etc.).
    *   *Historical Context:* Monopoly states -> Bipolar (Cold War) -> Unipolar/Multipolar now.
    *   *Analogy:* A game with rules, or an ecosystem.